# Asistente Virtual DUOC UC Plaza Oeste
Implementación de Arquitectura RAG con LangChain, FAISS y GitHub Models.

## Objetivo
Diseñar un asistente virtual basado en IA capaz de responder consultas académicas utilizando documentación institucional oficial.

In [ ]:
# INSTALAMOS EL PIP
!pip install langchain langchain-openai langchain-community faiss-cpu tiktoken pypdf python-dotenv

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN")
)

In [ ]:
# IMPORTACIONES IMPORTANTES

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
import os

In [ ]:
# NOS CONECTAMS CON EL MODELO Y HACEMOS UNA PRUEBA

from openai import OpenAI
import os

client = OpenAI(
    base_url=os.environ["GITHUB_BASE_URL"],
    api_key=os.environ["GITHUB_TOKEN"]
)

response = client.chat.completions.create(
    model="gpt-4o-mini",  # modelo disponible en GitHub Models
    messages=[
        {"role": "system", "content": "Eres un asistente académico."},
        {"role": "user", "content": "Explica qué es Rag en una frase."}
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

In [ ]:
# INTALAMOS LOS EMBEDDINGS

!pip install sentence-transformers

In [ ]:
# IMPORTAMOS DATA

import os

print(os.listdir("data"))

In [ ]:
# CARGAMOS LOS DOCUMENTOS CREADOS EN DATA

from langchain_community.document_loaders import PyPDFLoader
import os

def load_documents(data_path="data"):
    documents = []
    
    for file in os.listdir(data_path):
        if file.endswith(".pdf"):
            print(f"Cargando {file}...")
            loader = PyPDFLoader(os.path.join(data_path, file))
            documents.extend(loader.load())
    
    return documents

docs = load_documents()
print(f"\nSe cargaron {len(docs)} páginas en total.")

In [ ]:
# DIVIDIMOS EN CHUNKS

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=80
    )
    return text_splitter.split_documents(documents)

chunks = split_documents(docs)

print(f"Se generaron {len(chunks)} fragmentos.")

In [ ]:
# NOS ASEGURAMOS QUE LA API KEY ESTE FUNCIONADO BIEN
import os
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

In [ ]:
# CREAMOS EMBEDDINGS Y BASE VECTORIAL (FAISS)

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

def create_vectorstore_batched(chunks, batch_size=20):
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small"
    )
    
    vectorstore = None
    
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]
        print(f"Procesando batch {i//batch_size + 1}...")
        
        if vectorstore is None:
            vectorstore = FAISS.from_documents(batch, embeddings)
        else:
            vectorstore.add_documents(batch)
    
    return vectorstore

vectorstore = create_vectorstore_batched(chunks, batch_size=20)

print("Base vectorial creada correctamente.")

print("Base vectorial creada correctamente.")

In [ ]:
# PROBAMOS SI ENTREGA INFORMACIÓN DE UN DOCUMENTO

query = "¿Cómo puedo realizar mi matrícula?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\nResultado {i+1}:\n")
    print(doc.page_content[:500])

In [ ]:
# CREAMOS UN PROMPT OFICIAL

from langchain_core.prompts import PromptTemplate

template = """
Eres un asistente virtual del Instituto DUOC UC Plaza Oeste.
Debes responder exclusivamente utilizando la información entregada en el contexto.
Si la información no está presente, indica que dichos datos estan fuera de tu área de atención.
No inventes información.
Usa lenguaje formal y claro.

Contexto:
{context}

Pregunta del usuario:
{question}

Respuesta:
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [ ]:
# CONECTAMOS EL LLM

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3
)

In [ ]:
# VEMOS LA FUNCIÓN DEL RAG

def rag_answer(question):
    
    # 1. Recuperar documentos relevantes
    docs = vectorstore.similarity_search(question, k=3)
    
    # 2. Construir contexto
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # 3. Formatear prompt
    formatted_prompt = prompt.format(
        context=context,
        question=question
    )
    
    # 4. Llamar al modelo
    response = llm.invoke(formatted_prompt)
    
    return response.content

In [ ]:
# HACEMOS UNA PRUEBA DEL SISTEMA COMPLETO

respuesta = rag_answer("¿Cómo puedo realizar mi matrícula?")
print(respuesta)

In [ ]:
# HACEMOS UNA PREGUNTA FUERA DE CONTEXTO PARA VALIDAR ERRORES

respuesta = rag_answer("¿Cuál es la capital de Francia?")
print(respuesta)

In [ ]:
# GUARDAMOS LA BASE VECTORIAL

vectorstore.save_local("faiss_index")

In [ ]:
# PARA CARGAR LA BASE VECTORIAL A FUTURO

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)